# Datamine MOD2XYZ Process Example

This notebook demonstrates how to configure and run the **`mod2xyz`** process wrapper in `dmstudio`.

## Process Description

## MOD2XYZ

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

Assign field values from a block model to any file containing XYZ coordinate fields.

This can be useful for:

* assigning a rocktype value from a block model to a drillhole file
* assigning an interpolated grade from a block model to a drillhole file so that it can be compared with the actual sample value. In this case the field in one of the files must be renamed first before the process is run.
* assigning a value from a model to wireframe triangles. In order to achieve this the [COGTRI](<cogtri.md>) command should be run first to calculate the centre point of each triangle. Then the output file from **COGTRI** becomes the input file to **MOD2XYZ**.

The IN1 file is a standard block model file with one or more attribute fields. The IN2 file must include at least the three coordinate fields. The IN2 coordinate points are superimposed over the model cells and the selected field values (*F1, *F2, etc) are copied from the model cell to the sample point. The OUT file includes the same fields as the IN2 file plus the selected fields *F1, *F2, etc. If a sample point does not lie within a model cell then its field values, *F1, *F2, etc, will be set to absent data.

**Note** : This process supports **[retrieval criteria](<../COMMON/Retrieval_Criteria_Overview.md>)**.

### Input Files:

* **in1** (Block Model):
  Input model containing fields F1, F2, etc.
  Required=Yes

* **in2** (Undefined):
  Input file containing fields X, Y and Z
  Required=Yes

### Output Files:

* **out** (Undefined):
  Copy of IN2 file with extra fields F1, F2, etc from input model file.
  Required=Yes

### Fields:

* **x** (Numeric : IN2):
  X coordinate field in IN2 file.
  Default=Undefined
  Required=Yes

* **y** (Numeric : IN2):
  Y coordinate field in IN2 file.
  Default=Undefined
  Required=Yes

* **z** (Numeric : IN2):
  Z coordinate field in IN2 file.
  Default=Undefined
  Required=Yes

* **fields** (Undefined : Undefined):
  1st field in IN1 model to be copied to the OUT file.
  Default=Undefined
  Required=No

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('mod2xyz')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute mod2xyz
print("Running mod2xyz...")
dm_cmd.mod2xyz(
    inmods_i=['optional'],  # required
    out_o='t_mod2xyz_out',  # required
    x_f='X',  # required
    y_f='Y',  # required
    z_f='Z',  # required
    fields_f=['AU'],  # required
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("mod2xyz execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_mod2xyz_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")